In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [2]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [3]:
# How many lesson pages are there?

len(documents)

72

In [4]:
# Indexing and searching

from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)

In [5]:
# search based on a given question

question = "How does the agentic loop keep calling the model until it stops?"

search_results = index.search(
    question,
    # boost_dict={"question": 2.0, "section": 0.5},
    # filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

search_results[0]

{'content': '# The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we did function calling by hand. We sent a\nmessage and got back a function call. We ran it, sent the result back,\nand got the answer.\n\nThat works for one function call. It breaks down when the model wants\nto search several times, or when the first search misses the answer.\nWe don\'t know in advance how many calls the model will want. So we\nneed a loop that keeps calling the model and running tools until it\'s\ndone. An agent is exactly that.\n\n## Anatomy of an agent\n\nWith the LLM in the driver\'s seat, we have an agent. It\'s an AI\nassistant whose goal is to help the user.\n\nAn agent has three parts:\n\n- Instructions, the role and behavior we want. We pass this as the\n  `developer` message. The better the instructions, the better the\n  agent helps.\n- Tools, the functions the agent can call to carry 

In [6]:
search_results[0]['filename']

'01-agentic-rag/lessons/14-agentic-loop.md'

In [7]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [8]:
from rag_helper import RAGBase

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)

In [9]:
question = "How does the agentic loop keep calling the model until it stops?"
answer, usage = assistant.rag(question)
print(answer)

The loop keeps calling the model by using a `while True` loop and a `has_function_calls` flag.

How it works:
1. Call the model with the full message history.
2. Check the response for any `function_call` items.
3. If there is a function call, run the tool, append the tool result to `messages`, and set `has_function_calls = True`.
4. If there are no function calls in that response, break out of the loop.

So the agent stops only when the model returns a message with no more tool calls.


In [10]:
usage.input_tokens

7136

In [11]:
usage

ResponseUsage(input_tokens=7136, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=120, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=7256)

In [12]:
# Chunking

from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [13]:
# How many chunks?

len(chunks)

295

In [14]:
# Indexing the chunks

chunks_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

chunks_index.fit(chunks)

In [15]:
# querying the RAG with chunks index

assistant_chunks = RAGBase(
    index=chunks_index,
    llm_client=openai_client,
)

question = "How does the agentic loop keep calling the model until it stops?"
answer_chunks, usage_chunks = assistant_chunks.rag(question)
print(answer_chunks)

The loop keeps calling the model inside a `while True` loop, and after each call it checks whether the model returned any `function_call` items.

- If there is a function call, the code runs the tool, appends the tool result to `messages`, and continues.
- If there are no function calls in that turn, `has_function_calls` stays `False`, and the loop `break`s.

So the stop condition is: **no function calls this turn means the model is done**.


In [16]:
usage_chunks.input_tokens

2319

In [17]:
2319/7136

0.3249719730941704

In [19]:
# Qn 6: Turning it into an agent

from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [20]:
# Create a search function that uses the chunk index

def search(query: str) -> dict[str, str]:
    """
    Search the course lessons for entries matching the given query.
    """
    return chunks_index.search(
        query,
        num_results=5,
        boost_dict = {'filename': 2.0}
    )

In [21]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [22]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the course lessons for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [23]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

In [25]:
instructions = """
You're a course teaching assistant. 
Answer the student's question using the search tool. 
Make multiple searches with different keywords before answering.
"""

question = "How does the agentic loop work, and how is it different from plain RAG?"


In [26]:
runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model='gpt-5.4-mini')
)

In [28]:
result = runner.loop(
    prompt=question,
    callback=callback,
)

-> Response received


-> Response received


In [29]:
result = runner.loop(
    prompt=question,
    callback=callback,
)

-> Response received


-> Response received


In [30]:
result = runner.loop(
    prompt=question,
    callback=callback,
)

-> Response received


-> Response received
